# 06. Evaluation and Plots

This notebook evaluates all trained models and generates comprehensive plots.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').absolute()))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

from src.utils import set_seed, get_device, ensure_dir, save_json
from src.dataset import MultimodalFakeNewsDataset, Sample
from src.preprocessing import MultimodalPreprocessor
from src.classifier import MultimodalClassifier, ClassifierConfig
from src.evaluate import Evaluator, evaluate_models
from src.metrics import compute_metrics, print_metrics
from src.plotting import (
    plot_training_history, plot_validation_metrics,
    plot_confusion_matrix, plot_model_comparison,
    plot_clip_similarity_distribution, plot_ablation_study,
    plot_image_swap_comparison
)
from torch.utils.data import DataLoader

## Configuration

In [ ]:
set_seed(42)
device = get_device()

DATA_DIR = Path('data/processed')
IMAGES_DIR = Path('data/fakeddit/images')
CHECKPOINT_DIR = Path('artifacts/checkpoints')
RESULTS_DIR = Path('results')
PLOTS_DIR = RESULTS_DIR / 'plots'
ensure_dir(PLOTS_DIR)

## Load Test Data

In [ ]:
test_df = pd.read_csv(DATA_DIR / 'test.csv')

def create_samples(df, images_dir):
    samples = []
    for _, row in df.iterrows():
        text = str(row.get('clean_title', row.iloc[0]))
        image_id = str(row.get('image_id', ''))
        label = int(row.get('binary_label', row.iloc[-1]))
        
        image_path = images_dir / f"{image_id}.jpg" if image_id else None
        
        sample = Sample(
            sample_id=str(row.name),
            text=text,
            image_path=str(image_path) if image_path and image_path.exists() else None,
            label=label,
            metadata={}
        )
        samples.append(sample)
    return samples

test_samples = create_samples(test_df, IMAGES_DIR)
test_dataset = MultimodalFakeNewsDataset(test_samples)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4)

print(f"Test dataset: {len(test_dataset)} samples")

In [ ]:
preprocessor = MultimodalPreprocessor(device=device)

## Load All Models

In [ ]:
models = {}

def load_checkpoint(name, checkpoint_dir):
    checkpoint_path = checkpoint_dir / f"{name}_best.pt"
    if checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location=device)
        config = ClassifierConfig(**checkpoint['config']['model_config'])
        model = MultimodalClassifier(config)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
        model.eval()
        print(f"Loaded {name}")
        return model
    else:
        print(f"Checkpoint not found: {checkpoint_path}")
        return None

# Load available models
for name, subdir in [
    ('text_only', 'text_only'),
    ('image_only', 'image_only'),
    ('standard', 'standard'),
    ('consistency_aware', 'consistency_aware'),
    ('ablation', 'ablation')
]:
    model = load_checkpoint(name, CHECKPOINT_DIR / subdir)
    if model:
        models[name] = model

## Evaluate All Models

In [ ]:
all_results = {}

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Evaluating: {name}")
    print(f"{'='*50}")
    
    evaluator = Evaluator(model, preprocessor, device)
    results = evaluator.evaluate(test_loader)
    
    print_metrics(results['metrics'], f"{name} Results")
    
    all_results[name] = results

## Generate Plots

In [ ]:
# Model Comparison Plot
fig = plot_model_comparison(
    {name: res['metrics'] for name, res in all_results.items()},
    output_path=str(PLOTS_DIR / 'model_comparison.png'),
    title="All Models Comparison"
)
plt.show()

In [ ]:
# Confusion matrices for each model
for name, results in all_results.items():
    fig = plot_confusion_matrix(
        np.array(results['confusion_matrix']),
        output_path=str(PLOTS_DIR / f'{name}_confusion.png'),
        title=f'{name} Confusion Matrix'
    )
    plt.show()

In [ ]:
# CLIP similarity distribution for consistency-aware model
if 'consistency_aware' in all_results:
    fig = plot_clip_similarity_distribution(
        all_results['consistency_aware'],
        output_path=str(PLOTS_DIR / 'clip_similarity_dist.png'),
        title="CLIP Similarity Distribution"
    )
    plt.show()

## Save Results

In [ ]:
# Save all results
for name, results in all_results.items():
    save_json(results, RESULTS_DIR / f'{name}_results.json')

# Save summary comparison
summary = {
    name: res['metrics'] for name, res in all_results.items()
}
save_json(summary, RESULTS_DIR / 'all_models_summary.json')

print("Results saved to results/")

## Summary Statistics

In [ ]:
print("\n" + "="*70)
print("FINAL MODEL COMPARISON")
print("="*70)

metrics_to_show = ['accuracy', 'macro_f1', 'precision', 'recall', 'roc_auc']

header = f"{'Model':<25}" + "".join([f"{m:<12}" for m in metrics_to_show])
print(header)
print("-" * 85)

for name in sorted(all_results.keys()):
    metrics = all_results[name]['metrics']
    row = f"{name:<25}"
    for m in metrics_to_show:
        val = metrics.get(m, 0)
        row += f"{val:.4f}     "
    print(row)

print("-" * 85)

In [ ]:
print("\n" + "="*70)
print("EVALUATION COMPLETE")
print("="*70)

print(f"\nModels evaluated: {len(all_results)}")
print(f"Test samples: {len(test_dataset)}")

best_model = max(all_results.keys(), key=lambda k: all_results[k]['metrics']['macro_f1'])
print(f"\nBest model (by Macro F1): {best_model}")
print(f"  Accuracy: {all_results[best_model]['metrics']['accuracy']:.4f}")
print(f"  Macro F1: {all_results[best_model]['metrics']['macro_f1']:.4f}")

print("\nPlots saved to:", PLOTS_DIR)
print("\nYou can now run the Streamlit app: streamlit run app.py")